
# Sentinel-2 Satellite Land-Cover Classification Project

This notebook demonstrates a complete machine learning workflow to classify Sentinel-2 satellite imagery into five land-cover classes:
1. Bare/Sparse Vegetation
2. Built-Up
3. Cropland
4. Permanent Water
5. Tree Cover

## Project Setup and Google Drive Integration
We first mount Google Drive to access our persistent project directory and datasets.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
import kagglehub

# Download latest version of the dataset from Kaggle
kaggle_path = kagglehub.dataset_download("zimransohail/satellite-land-cover-classification")
print("Dataset downloaded to:", kaggle_path)

100%|██████████| 73.7M/73.7M [00:01<00:00, 55.3MB/s]

Extracting files...


Dataset downloaded to: /root/.cache/kagglehub/datasets/zimransohail/satellite-land-cover-classification/versions/1


In [2]:
import os

# List the contents of the downloaded directory to see how files are organized
if os.path.exists(kaggle_path):
    print(os.listdir(kaggle_path))
else:
    print("Path not found.")

['dataset', 'metadata']


In [4]:
import os

paths_to_verify = {
    "DATASET_64_DIR": DATASET_64_DIR,
    "DATASET_128_DIR": DATASET_128_DIR,
    "DATASET_128_ALIGNED_DIR": DATASET_128_ALIGNED_DIR,
    "KAGGLE_DATASET_ROOT": KAGGLE_DATASET_ROOT
}

print("=== VERIFYING GLOBAL DIRECTORY PATHS ===")
for name, path in paths_to_verify.items():
    exists = os.path.exists(path)
    print(f"{name}: {'[FOUND]' if exists else '[MISSING]'} -> {path}")
    if exists and os.path.isdir(path):
        sub_items = os.listdir(path)
        print(f"  Contains {len(sub_items)} items: {sub_items[:5]}...")

=== VERIFYING GLOBAL DIRECTORY PATHS ===
DATASET_64_DIR: [FOUND] -> /root/.cache/kagglehub/datasets/zimransohail/satellite-land-cover-classification/versions/1/dataset/landcover_dataset
  Contains 3 items: ['val', 'train', 'test']...
DATASET_128_DIR: [FOUND] -> /root/.cache/kagglehub/datasets/zimransohail/satellite-land-cover-classification/versions/1/dataset/landcover_dataset_128
  Contains 3 items: ['val', 'train', 'test']...
DATASET_128_ALIGNED_DIR: [FOUND] -> /root/.cache/kagglehub/datasets/zimransohail/satellite-land-cover-classification/versions/1/dataset/landcover_dataset_128_aligned
  Contains 3 items: ['val', 'train', 'test']...
KAGGLE_DATASET_ROOT: [FOUND] -> /root/.cache/kagglehub/datasets/zimransohail/satellite-land-cover-classification/versions/1/dataset
  Contains 3 items: ['landcover_dataset', 'landcover_dataset_128_aligned', 'landcover_dataset_128']...


## Section 1: Environment & Global Configuration
### Subsection 1.1: Library Imports & Global Seeds


## Global Configurations & Library Imports
To prevent duplication, all required libraries, global path variables, and device settings are centralized here.


In [3]:
import os
import copy
import time
import json
import random
import shutil
import zipfile
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import requests

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Configure global seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Set device configurations
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Execution Device:", device)
if device.type == "cuda":
    print("GPU Model:", torch.cuda.get_device_name(0))

# Define project directories - Migrated from Drive to Kagglehub destination
PROJECT_DIR = "/content/drive/MyDrive/landcover_project"
KAGGLE_DATASET_ROOT = "/root/.cache/kagglehub/datasets/zimransohail/satellite-land-cover-classification/versions/1/dataset"

DATASET_64_DIR = os.path.join(KAGGLE_DATASET_ROOT, "landcover_dataset")
DATASET_128_DIR = os.path.join(KAGGLE_DATASET_ROOT, "landcover_dataset_128")
DATASET_128_ALIGNED_DIR = os.path.join(KAGGLE_DATASET_ROOT, "landcover_dataset_128_aligned")

classes = [
    "Bare_Sparse_Vegetation",
    "Built_Up",
    "Cropland",
    "Permanent_Water",
    "Tree_Cover"
]


Execution Device: cuda
GPU Model: Tesla T4


## Section 2: Dataset Verification (64x64)
### Subsection 2.1: Geographic Splitting Verification

# Land-Cover Dataset Verification

This section checks the downloaded Sentinel-2 image dataset saved in Google Drive. It verifies that each PNG can be opened and has the expected dimensions (64×64) and RGB mode.

**Before running:** In Google Colab, run the Drive-mount cell first, then run the verification cell. This notebook only reads/checks files; it does not download, delete, or modify images.

In [ ]:
# Check that the dataset folder is available before scanning.
if not os.path.isdir(DATASET_64_DIR):
    raise FileNotFoundError(f"Dataset folder not found: {DATASET_64_DIR}\nCheck that Google Drive is mounted and the path is correct.")

splits = ["train", "val", "test"]
results = []
missing_folders = []

for split in splits:
    for class_name in classes:
        folder = os.path.join(DATASET_64_DIR, split, class_name)
        if not os.path.isdir(folder):
            missing_folders.append(folder)
            results.append({
                "Split": split,
                "Class": class_name,
                "Valid images": 0,
                "Invalid images": 0,
                "Folder status": "MISSING"
            })
            continue

        valid = 0
        invalid = 0

        for filename in os.listdir(folder):
            if not filename.lower().endswith('.png'):
                continue

            path = os.path.join(folder, filename)
            try:
                with Image.open(path) as img:
                    img.verify()
                with Image.open(path) as img:
                    if img.size == (64, 64) and img.mode == 'RGB':
                        valid += 1
                    else:
                        invalid += 1
            except Exception:
                invalid += 1

        results.append({
            "Split": split,
            "Class": class_name,
            "Valid images": valid,
            "Invalid images": invalid,
            "Folder status": "OK"
        })

results_df = pd.DataFrame(results)
display(results_df)

print("\nTotal valid images:", results_df["Valid images"].sum())
print("Total invalid images:", results_df["Invalid images"].sum())
print("\nImages per split:")
print(results_df.groupby("Split")["Valid images"].sum())

if missing_folders:
    print("\nWARNING: These folders are missing:")
    for folder in missing_folders:
        print(folder)

#Visualize a random sample
### This cell displays random images from the dataset with their class and split. Use it to check for blank images, heavy cloud cover, unusual colors, or obvious label mismatches. It only reads images and does not modify them.

In [ ]:
# Choose how many images to display from each split.
SAMPLES_PER_CLASS = 2  # 2 per class per split = up to 30 images
splits = ["train", "val", "test"]
samples = []

for split in splits:
    for class_name in classes:
        folder = os.path.join(DATASET_64_DIR, split, class_name)
        files = [
            os.path.join(folder, f)
            for f in os.listdir(folder)
            if f.lower().endswith(".png")
        ]
        chosen = random.sample(files, min(SAMPLES_PER_CLASS, len(files)))
        for path in chosen:
            samples.append((path, split, class_name))

# Display a grid of the selected images.
ncols = 5
nrows = (len(samples) + ncols - 1) // ncols
fig, axes = plt.subplots(nrows, ncols, figsize=(15, 3 * nrows))
axes = axes.flatten() if hasattr(axes, "flatten") else [axes]

for ax, (path, split, class_name) in zip(axes, samples):
    with Image.open(path) as img:
        ax.imshow(img)
    ax.set_title(f"{class_name}\n{split}", fontsize=9)
    ax.axis("off")

for ax in axes[len(samples):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

### Subsection 2.2: System & Package Diagnostics

#Optional: inspect more images
###If the first grid looks reasonable, change SAMPLES_PER_CLASS = 2 to 4 and rerun the visualization cell. That displays up to 60 images. You can also change random.seed(42) to another integer to sample different images.

In [ ]:
# Display library and system environment information
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__ if 'torchvision' in globals() else "Loaded")
print("Scikit-learn:", sklearn.__version__ if 'sklearn' in globals() else "Loaded")
print("Execution Device:", device)
if device.type == "cuda":
    print("GPU Model:", torch.cuda.get_device_name(0))

## Section 3: Training ResNet18 from Scratch (64x64)
### Subsection 3.1: Preprocessing and Transforms Setup

In [ ]:
# ImageNet normalization, suitable for our pretrained ResNet18 experiment
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# Training: resize and apply light data augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    normalize
])

# Validation and testing: no random augmentation
eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])

# Load each split using the global DATASET_64_DIR
train_dataset = datasets.ImageFolder(
    os.path.join(DATASET_64_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATASET_64_DIR, "val"),
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    os.path.join(DATASET_64_DIR, "test"),
    transform=eval_transform
)

# Ensure all splits use the same class-to-index mapping
assert train_dataset.class_to_idx == val_dataset.class_to_idx
assert train_dataset.class_to_idx == test_dataset.class_to_idx

print("Classes:", train_dataset.classes)
print("Class mapping:", train_dataset.class_to_idx)
print()
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))
print("Total 64x64 images:", len(train_dataset) + len(val_dataset) + len(test_dataset))

### Subsection 3.2: DataLoaders Implementation

In [ ]:
BATCH_SIZE = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

# Inspect one batch
images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch label shape:", labels.shape)
print("Number of training batches:", len(train_loader))
print("Number of validation batches:", len(val_loader))
print("Number of test batches:", len(test_loader))

### Subsection 3.3: Model Initialization (ResNet18 Scratch)

In [ ]:
NUM_CLASSES = len(train_dataset.classes)

# Create ResNet18 with randomly initialized weights
model = models.resnet18(weights=None)

# Replace the final classification layer for our 5 classes
model.fc = nn.Linear(
    model.fc.in_features,
    NUM_CLASSES
)

# Move the model to the GPU
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(
    p.numel() for p in model.parameters() if p.requires_grad
)

print("Model: ResNet18 (trained from scratch)")
print("Pretrained weights: None")
print("Number of classes:", NUM_CLASSES)
print("Class names:", train_dataset.classes)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
print("Model device:", next(model.parameters()).device)

### Subsection 3.4: Optimizer and Loss Setup

In [ ]:
criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=0.001
)

EPOCHS = 15

print("Loss function:", criterion)
print("Optimizer: Adam")
print("Learning rate: 0.001")
print("Epochs:", EPOCHS)

### Subsection 3.5: Training & Validation Loop

In [ ]:
best_val_accuracy = 0.0
best_model_state = copy.deepcopy(model.state_dict())

history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(EPOCHS):
    start_time = time.time()

    # --------------------
    # Training
    # --------------------
    model.train()

    running_loss = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss = running_loss / train_total
    train_accuracy = train_correct / train_total

    # --------------------
    # Validation
    # --------------------
    model.eval()

    val_running_loss = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_running_loss += loss.item() * images.size(0)

            predictions = outputs.argmax(dim=1)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_running_loss / val_total
    val_accuracy = val_correct / val_total

    history["train_loss"].append(train_loss)
    history["train_accuracy"].append(train_accuracy)
    history["val_loss"].append(val_loss)
    history["val_accuracy"].append(val_accuracy)

    # Save the best model based on validation accuracy
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_model_state = copy.deepcopy(model.state_dict())

    elapsed = time.time() - start_time

    print(
        f"Epoch [{epoch + 1:02d}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

# Restore the best-performing validation checkpoint
model.load_state_dict(best_model_state)

print(f"\nBest validation accuracy: {best_val_accuracy:.4f}")

### Subsection 3.6: Performance Metrics & Loss Visualization

In [ ]:
epochs = range(1, len(history["train_loss"]) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history["train_loss"], marker="o", label="Training")
plt.plot(epochs, history["val_loss"], marker="o", label="Validation")
plt.title("Loss Across Epochs")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
plt.plot(
    epochs,
    [x * 100 for x in history["train_accuracy"]],
    marker="o",
    label="Training"
)
plt.plot(
    epochs,
    [x * 100 for x in history["val_accuracy"]],
    marker="o",
    label="Validation"
)
plt.title("Accuracy Across Epochs")
plt.xlabel("Epoch")
plt.ylabel("Accuracy (%)")
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

### Subsection 3.7: Checkpointing Best Model

In [ ]:
best_epoch = max(
    range(len(history["val_accuracy"])),
    key=lambda i: history["val_accuracy"][i]
) + 1

print("Best epoch:", best_epoch)
print(f"Best validation accuracy: {best_val_accuracy * 100:.2f}%")

# Save the best checkpoint to Google Drive
checkpoint_path = (
    "/content/drive/MyDrive/landcover_project/"
    "resnet18_scratch_best.pth"
)

torch.save(
    {
        "model_state_dict": best_model_state,
        "class_to_idx": train_dataset.class_to_idx,
        "best_epoch": best_epoch,
        "best_val_accuracy": best_val_accuracy,
    },
    checkpoint_path
)

print("Checkpoint saved to:", checkpoint_path)

### Subsection 3.8: Validation Performance Report & Confusion Matrix

In [ ]:
# Use the best checkpoint already restored in `model`
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        predictions = outputs.argmax(dim=1)

        all_predictions.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

# Calculate overall accuracy
accuracy = accuracy_score(all_labels, all_predictions)
print(f"Validation Accuracy: {accuracy * 100:.2f}%")

# Per-class precision, recall, and F1-score
class_names = train_dataset.classes

print("\nClassification Report:\n")
print(
    classification_report(
        all_labels,
        all_predictions,
        labels=list(range(len(class_names))),
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

# Confusion matrix
cm = confusion_matrix(
    all_labels,
    all_predictions,
    labels=list(range(len(class_names)))
)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted Class")
plt.ylabel("Actual Class")
plt.title("ResNet18 Scratch — Validation Confusion Matrix")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### Subsection 3.9: Exporting Phase 1 Summary

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/landcover_project"
os.makedirs(PROJECT_DIR, exist_ok=True)

# Save training history
with open(
    os.path.join(PROJECT_DIR, "scratch_training_history.json"),
    "w"
) as f:
    json.dump(history, f, indent=4)

# Save experiment summary
scratch_summary = {
    "model": "ResNet18",
    "initialization": "random",
    "epochs": EPOCHS,
    "best_epoch": best_epoch,
    "best_validation_accuracy": best_val_accuracy,
    "train_images": len(train_dataset),
    "validation_images": len(val_dataset),
    "test_images": len(test_dataset),
    "classes": train_dataset.classes
}

with open(
    os.path.join(PROJECT_DIR, "scratch_model_summary.json"),
    "w"
) as f:
    json.dump(scratch_summary, f, indent=4)

print("Scratch-model experiment records saved.")

## Section 4: Transfer Learning Phase (Frozen Backbone on 128x128)
### Subsection 4.1: Dataset & Verification (128x128)

In [ ]:
# Restore the best weights from the original 15 epochs
model.load_state_dict(best_model_state)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0003,
    weight_decay=0.0001
)

scheduler = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2
)

FINE_TUNE_EPOCHS = 20
PATIENCE = 5

# Track the best result across both training phases
best_ft_accuracy = best_val_accuracy
best_ft_loss = float("inf")
best_ft_state = copy.deepcopy(model.state_dict())
epochs_without_improvement = 0

fine_tune_history = {
    "train_loss": [],
    "train_accuracy": [],
    "val_loss": [],
    "val_accuracy": []
}

for epoch in range(FINE_TUNE_EPOCHS):
    start_time = time.time()

    # Training
    model.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * labels.size(0)
        train_correct += (outputs.argmax(1) == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss_sum / train_total
    train_accuracy = train_correct / train_total

    # Validation
    model.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss_sum += loss.item() * labels.size(0)
            val_correct += (outputs.argmax(1) == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_accuracy = val_correct / val_total

    scheduler.step(val_loss)

    fine_tune_history["train_loss"].append(train_loss)
    fine_tune_history["train_accuracy"].append(train_accuracy)
    fine_tune_history["val_loss"].append(val_loss)
    fine_tune_history["val_accuracy"].append(val_accuracy)

    # Keep the best checkpoint; use lower validation loss to break ties
    improved = (
        val_accuracy > best_ft_accuracy
        or (
            val_accuracy == best_ft_accuracy
            and val_loss < best_ft_loss
        )
    )

    if improved:
        best_ft_accuracy = val_accuracy
        best_ft_loss = val_loss
        best_ft_state = copy.deepcopy(model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    elapsed = time.time() - start_time
    current_lr = optimizer.param_groups[0]["lr"]

    print(
        f"Fine-tune [{epoch + 1:02d}/{FINE_TUNE_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_accuracy:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"LR: {current_lr:.6f} | "
        f"Time: {elapsed:.1f}s"
    )

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping: validation accuracy did not improve.")
        break

# Restore the best checkpoint from fine-tuning
model.load_state_dict(best_ft_state)

# Update the best checkpoint variables for subsequent cells
best_model_state = copy.deepcopy(best_ft_state)
best_val_accuracy = best_ft_accuracy

print(f"\nBest validation accuracy: {best_ft_accuracy * 100:.2f}%")

##the validation accuracy is good but more can be achived with a better 128 x 128 dataset. So another experiment will be run with a higher image resoultion

In [ ]:
# Verification of the 128x128 dataset
splits = ["train", "val", "test"]

for split in splits:
    split_dir = os.path.join(DATASET_128_DIR, split)
    print(f"\n{split.upper()}")

    total = 0
    for cls in sorted(os.listdir(split_dir)):
        class_dir = os.path.join(split_dir, cls)

        if not os.path.isdir(class_dir):
            continue

        images = [
            f for f in os.listdir(class_dir)
            if f.lower().endswith((".png", ".jpg", ".jpeg"))
        ]

        total += len(images)

        if images:
            img = Image.open(os.path.join(class_dir, images[0]))
            print(f"{cls}: {len(images)} images | sample size: {img.size}")

    print("Total:", total)

### Subsection 4.2: Aligned Pipeline Datasets Creation

In [ ]:
BATCH_SIZE = 32
NUM_WORKERS = 2

train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    normalize
])

eval_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    normalize
])

train_dataset = datasets.ImageFolder(
    os.path.join(DATASET_128_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATASET_128_DIR, "val"),
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    os.path.join(DATASET_128_DIR, "test"),
    transform=eval_transform
)

assert train_dataset.class_to_idx == val_dataset.class_to_idx
assert train_dataset.class_to_idx == test_dataset.class_to_idx

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE,
    shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE,
    shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available()
)

print("Classes:", train_dataset.class_to_idx)
print("Train images:", len(train_dataset))
print("Validation images:", len(val_dataset))

In [ ]:
print("Class mapping:", train_dataset.class_to_idx)

images, labels = next(iter(train_loader))

print("Batch shape:", images.shape)
print("Label shape:", labels.shape)
print("Image tensor dtype:", images.dtype)
print("Label range:", labels.min().item(), "to", labels.max().item())

assert images.shape[1:] == (3, 128, 128)
assert len(train_dataset.classes) == 5

print("Dataset checks passed!")

In [ ]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 5)
model = model.to(device)

print("Device:", device)
print("Input resolution: 128x128")

In [ ]:
paths_to_check = [
    os.path.join(PROJECT_DIR, "landcover_split_metadata.csv"),
    os.path.join(PROJECT_DIR, "landcover_split_metadata_128.csv"),
    os.path.join(PROJECT_DIR, "selected_candidates.csv"),
    DATASET_64_DIR,
    DATASET_128_DIR,
]

for path in paths_to_check:
    print(("FOUND  " if os.path.exists(path) else "MISSING"), path)

In [ ]:
meta64_path = os.path.join(PROJECT_DIR, "landcover_split_metadata.csv")
meta128_path = os.path.join(PROJECT_DIR, "landcover_split_metadata_128.csv")

meta64 = pd.read_csv(meta64_path)
meta128 = pd.read_csv(meta128_path)

print("64x64 metadata shape:", meta64.shape)
print("128x128 metadata shape:", meta128.shape)

print("\n64x64 columns:", meta64.columns.tolist())
print("128x128 columns:", meta128.columns.tolist())

In [ ]:
# 1. Compare split counts
print("64x64 split counts:")
print(pd.crosstab(meta64["split"], meta64["Class"]))

print("\n128x128 split counts:")
print(pd.crosstab(meta128["split"], meta128["Class"]))

# 2. Check whether class, coordinates, and geographic groups match
compare_cols = [
    "class_id", "Class", "Latitude", "Longitude", "geo_group"
]

a = meta64[compare_cols].copy().reset_index(drop=True)
b = meta128[compare_cols].copy().reset_index(drop=True)

print("\nSame row order and locations:", a.equals(b))

# 3. Check split assignments row by row
print(
    "Rows with different split assignments:",
    (meta64["split"].reset_index(drop=True) !=
     meta128["split"].reset_index(drop=True)).sum()
)

# 4. Check if every geographic group stays within one split
for name, df in [("64x64", meta64), ("128x128", meta128)]:
    groups_per_split = df.groupby("geo_group")["split"].nunique()
    print(f"{name} groups appearing in multiple splits:",
          (groups_per_split > 1).sum())

In [ ]:
# Compare the location/class records independent of row order
key_cols = ["class_id", "Latitude", "Longitude", "geo_group"]

a = meta64[key_cols].copy()
b = meta128[key_cols].copy()

# Round coordinates to avoid insignificant floating-point differences
for df in (a, b):
    df["Latitude"] = df["Latitude"].round(6)
    df["Longitude"] = df["Longitude"].round(6)

a_keys = set(map(tuple, a.to_numpy()))
b_keys = set(map(tuple, b.to_numpy()))

print("Locations in 64x64:", len(a_keys))
print("Locations in 128x128:", len(b_keys))
print("Common locations:", len(a_keys & b_keys))
print("Only in 64x64:", len(a_keys - b_keys))
print("Only in 128x128:", len(b_keys - a_keys))

In [ ]:
def make_key(df):
    return (
        df["class_id"].astype(str) + "_" +
        df["Latitude"].round(6).astype(str) + "_" +
        df["Longitude"].round(6).astype(str)
    )

meta64["match_key"] = make_key(meta64)
meta128["match_key"] = make_key(meta128)

assert meta64["match_key"].is_unique
assert meta128["match_key"].is_unique
assert set(meta64["match_key"]) == set(meta128["match_key"])

# Transfer the original split assignments
original_splits = meta64.set_index("match_key")["split"]
meta128["split"] = meta128["match_key"].map(original_splits)

assert meta128["split"].notna().all()

# Verify geographic groups remain separated
assert (
    meta128.groupby("geo_group")["split"].nunique().max() == 1
), "A geographic group appears in multiple splits!"

# Save aligned metadata
meta128 = meta128.drop(columns=["match_key"])
aligned_path = os.path.join(PROJECT_DIR, "landcover_split_metadata_128_aligned.csv")
meta128.to_csv(aligned_path, index=False)

print("Aligned metadata saved:", aligned_path)
print("\nAligned split counts:")
print(pd.crosstab(meta128["split"], meta128["Class"]))
print("\nSplit assignments match original:", meta128["split"].reset_index(drop=True).equals(meta64["split"].reset_index(drop=True)))

In [ ]:
import os
import shutil
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/landcover_project"

SOURCE_DIR = os.path.join(PROJECT_DIR, "landcover_dataset_128")
OUTPUT_DIR = os.path.join(PROJECT_DIR, "landcover_dataset_128_aligned")
META_PATH = os.path.join(
    PROJECT_DIR, "landcover_split_metadata_128_aligned.csv"
)

meta = pd.read_csv(META_PATH)

# Build a stable key for matching metadata rows to image files
def make_key(df):
    return (
        df["class_id"].astype(str) + "_" +
        df["Latitude"].round(6).astype(str) + "_" +
        df["Longitude"].round(6).astype(str)
    )

meta["match_key"] = make_key(meta)

# Find each image using its class folder and filename
# Original image folders use ImageFolder-compatible class names.
source_files = {}

for split in ["train", "val", "test"]:
    for class_name in meta["Class"].unique():
        folder = os.path.join(SOURCE_DIR, split, class_name)
        if not os.path.isdir(folder):
            continue

        for filename in os.listdir(folder):
            if filename.lower().endswith((".png", ".jpg", ".jpeg")):
                path = os.path.join(folder, filename)
                source_files.setdefault(filename, []).append(path)

print("Metadata rows:", len(meta))
print("Unique source filenames:", len(source_files))

In [ ]:
for split in ["train", "val", "test"]:
    for class_name in sorted(meta["Class"].unique()):
        folder = os.path.join(SOURCE_DIR, split, class_name)
        if os.path.isdir(folder):
            print(f"\n{split}/{class_name}")
            print(os.listdir(folder)[:5])
            break
    break

print("\n128x128 metadata columns:", meta.columns.tolist())
print(meta[["Class", "Latitude", "Longitude", "geo_group", "split"]].head())

In [ ]:
meta64 = pd.read_csv(os.path.join(PROJECT_DIR, "landcover_split_metadata.csv"))
meta128 = pd.read_csv(os.path.join(PROJECT_DIR, "landcover_split_metadata_128_aligned.csv"))

def make_key(df):
    return (
        df["class_id"].astype(str) + "_" +
        df["Latitude"].round(6).astype(str) + "_" +
        df["Longitude"].round(6).astype(str)
    )

meta64["match_key"] = make_key(meta64)
meta128["match_key"] = make_key(meta128)

image_map = meta64.set_index("match_key")["image_id"].to_dict()
meta128["image_id"] = meta128["match_key"].map(image_map)

assert meta128["image_id"].notna().all()
assert meta128["image_id"].is_unique

source_map = {}
for root, _, files in os.walk(DATASET_128_DIR):
    for filename in files:
        if filename.lower().endswith((".png", ".jpg", ".jpeg")):
            if filename in source_map:
                raise ValueError(f"Duplicate filename found: {filename}")
            source_map[filename] = os.path.join(root, filename)

assert len(source_map) == 2500, f"Expected 2500 images, found {len(source_map)}"

if os.path.exists(DATASET_128_ALIGNED_DIR):
    raise FileExistsError(f"{DATASET_128_ALIGNED_DIR} already exists. Rename or remove it manually.")

os.makedirs(DATASET_128_ALIGNED_DIR)
copied = 0

for _, row in meta128.iterrows():
    filename = f"{row['image_id']}.png"
    if filename not in source_map:
        raise FileNotFoundError(f"Image not found: {filename}")

    destination_folder = os.path.join(DATASET_128_ALIGNED_DIR, row["split"], row["Class"])
    os.makedirs(destination_folder, exist_ok=True)
    shutil.copy2(source_map[filename], os.path.join(destination_folder, filename))
    copied += 1

print("Images copied:", copied)
print("Aligned dataset:", DATASET_128_ALIGNED_DIR)

### Subsection 4.3: Aligned Pipeline Check & Verifications

In [ ]:
aligned_train = datasets.ImageFolder(os.path.join(DATASET_128_ALIGNED_DIR, "train"))
aligned_val = datasets.ImageFolder(os.path.join(DATASET_128_ALIGNED_DIR, "val"))
aligned_test = datasets.ImageFolder(os.path.join(DATASET_128_ALIGNED_DIR, "test"))

for name, dataset in [("TRAIN", aligned_train), ("VALIDATION", aligned_val), ("TEST", aligned_test)]:
    print(f"\n{name}: {len(dataset)} images")
    print(dataset.class_to_idx)

    for path, label in dataset.samples:
        with Image.open(path) as img:
            assert img.size == (128, 128), f"Unexpected image size: {path}, {img.size}"
            assert img.mode == "RGB", f"Unexpected image mode: {path}, {img.mode}"

print("\nExpected totals: train=1766, val=367, test=367")
assert len(aligned_train) == 1766
assert len(aligned_val) == 367
assert len(aligned_test) == 367
print("Verification passed.")

### Subsection 4.4: Model Training Configuration (128x128)

### 128x128 training model section

In [ ]:
CHECKPOINT_PATH = os.path.join(PROJECT_DIR, "resnet18_128_aligned_best.pth")
HISTORY_PATH = os.path.join(PROJECT_DIR, "resnet18_128_aligned_history.csv")

NUM_EPOCHS = 20
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
PATIENCE = 5

model_128 = models.resnet18(weights=None)
model_128.fc = nn.Linear(model_128.fc.in_features, 5)
model_128 = model_128.to(device)

criterion_128 = nn.CrossEntropyLoss()
optimizer_128 = optim.AdamW(
    model_128.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

scheduler_128 = optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_128,
    mode="max",
    factor=0.5,
    patience=2
)

print("Device:", device)
print("Experiment: ResNet18 trained from scratch, 128x128")
print("Checkpoint:", CHECKPOINT_PATH)
print("History:", HISTORY_PATH)

### Subsection 4.5: Training & Tuning Loop

In [ ]:
history_128 = []
best_val_acc_128 = 0.0
best_epoch_128 = 0
epochs_without_improvement = 0
best_model_weights_128 = copy.deepcopy(model_128.state_dict())

start_time = time.time()

for epoch in range(NUM_EPOCHS):
    # ---------------- TRAIN ----------------
    model_128.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer_128.zero_grad()

        outputs = model_128(images)
        loss = criterion_128(outputs, labels)

        loss.backward()
        optimizer_128.step()

        train_loss_sum += loss.item() * images.size(0)
        predictions = outputs.argmax(dim=1)
        train_correct += (predictions == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    # ---------------- VALIDATION ----------------
    model_128.eval()

    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model_128(images)
            loss = criterion_128(outputs, labels)

            val_loss_sum += loss.item() * images.size(0)
            predictions = outputs.argmax(dim=1)
            val_correct += (predictions == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    # Adjust learning rate using validation accuracy
    scheduler_128.step(val_acc)

    current_lr = optimizer_128.param_groups[0]["lr"]

    history_128.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "learning_rate": current_lr
    })

    # Save best model based on validation accuracy
    if val_acc > best_val_acc_128:
        best_val_acc_128 = val_acc
        best_epoch_128 = epoch + 1
        best_model_weights_128 = copy.deepcopy(
            model_128.state_dict()
        )

        torch.save({
            "epoch": best_epoch_128,
            "model_state_dict": best_model_weights_128,
            "optimizer_state_dict": optimizer_128.state_dict(),
            "best_val_accuracy": best_val_acc_128,
            "class_to_idx": train_loader.dataset.class_to_idx,
            "input_size": 128
        }, CHECKPOINT_PATH)

        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1

    # Save history after every epoch in case Colab disconnects
    pd.DataFrame(history_128).to_csv(HISTORY_PATH, index=False)

    print(
        f"Epoch [{epoch + 1:02d}/{NUM_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"LR: {current_lr:.6f}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"Early stopping at epoch {epoch + 1}.")
        break

# Restore the best validation checkpoint in memory
model_128.load_state_dict(best_model_weights_128)

elapsed = (time.time() - start_time) / 60

print("\nTraining complete.")
print(f"Best validation accuracy: {best_val_acc_128:.4f}")
print(f"Best epoch: {best_epoch_128}")
print(f"Training time: {elapsed:.2f} minutes")
print("Best checkpoint saved:", CHECKPOINT_PATH)
print("History saved:", HISTORY_PATH)

### Subsection 4.6: Loss & Accuracy Plotting (128x128)

In [ ]:
import matplotlib.pyplot as plt

history_df_128 = pd.DataFrame(history_128)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(
    history_df_128["epoch"],
    history_df_128["train_loss"],
    label="Train loss"
)
axes[0].plot(
    history_df_128["epoch"],
    history_df_128["val_loss"],
    label="Validation loss"
)
axes[0].set_title("128x128 ResNet18 — Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Loss")
axes[0].legend()

axes[1].plot(
    history_df_128["epoch"],
    history_df_128["train_accuracy"],
    label="Train accuracy"
)
axes[1].plot(
    history_df_128["epoch"],
    history_df_128["val_accuracy"],
    label="Validation accuracy"
)
axes[1].set_title("128x128 ResNet18 — Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Accuracy")
axes[1].legend()

plt.tight_layout()
plt.show()

### Subsection 4.7: Testing Metrics & Diagnostic Matrix

In [ ]:
# Load the saved best checkpoint
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location=device,
    weights_only=False
)

model_128.load_state_dict(checkpoint["model_state_dict"])
model_128.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader:
        images = images.to(device)
        outputs = model_128(images)
        predictions = outputs.argmax(dim=1)

        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.numpy())

test_accuracy = accuracy_score(all_labels, all_preds)

class_names = [
    name for name, index
    in sorted(test_loader.dataset.class_to_idx.items(),
              key=lambda item: item[1])
]

print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
print(f"Checkpoint epoch: {checkpoint["epoch"]}")
print("\nClassification Report:\n")
print(classification_report(
    all_labels,
    all_preds,
    target_names=class_names,
    digits=4
))

cm = confusion_matrix(
    all_labels,
    all_preds,
    labels=list(range(len(class_names)))
)

plt.figure(figsize=(9, 7))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=class_names,
    yticklabels=class_names
)
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.title("128x128 ResNet18 — Test Confusion Matrix")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

### Subsection 4.8: Pretrained Weights Sourcing & Setup

In [ ]:
from collections import Counter

print("Test images:", len(test_loader.dataset))
print("Dataset path:", test_loader.dataset.root)

print("\nClass counts:")
print(Counter(
    test_loader.dataset.classes[label]
    for _, label in test_loader.dataset.samples
))

In [ ]:
# 1. Correct aligned test dataset
aligned_test_dir = os.path.join(DATASET_128_ALIGNED_DIR, "test")

test_dataset_aligned = datasets.ImageFolder(
    aligned_test_dir,
    transform=eval_transform
)

test_loader_aligned = DataLoader(
    test_dataset_aligned,
    batch_size=32,
    shuffle=False
)

print("Aligned test images:", len(test_dataset_aligned))
print("Class mapping:", test_dataset_aligned.class_to_idx)

class_counts = Counter(
    test_dataset_aligned.classes[label]
    for _, label in test_dataset_aligned.samples
)
print("Class counts:", class_counts)

assert len(test_dataset_aligned) == 367, "Expected 367 aligned test images. Check the dataset path."

# 2. Recreate the model and load the saved checkpoint
model_128 = models.resnet18(weights=None)
model_128.fc = nn.Linear(model_128.fc.in_features, 5)

checkpoint_path = os.path.join(PROJECT_DIR, "resnet18_128_aligned_best.pth")
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)

if "model_state_dict" in checkpoint:
    model_128.load_state_dict(checkpoint["model_state_dict"])
elif "state_dict" in checkpoint:
    model_128.load_state_dict(checkpoint["state_dict"])
else:
    model_128.load_state_dict(checkpoint)

model_128 = model_128.to(device)
model_128.eval()

# 3. Evaluate on the aligned test set
all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader_aligned:
        images = images.to(device)
        outputs = model_128(images)
        predictions = outputs.argmax(dim=1).cpu().numpy()
        all_preds.extend(predictions)
        all_labels.extend(labels.numpy())

test_accuracy = accuracy_score(all_labels, all_preds)

print(f"\nAligned Test Accuracy: {test_accuracy:.4%}")
print("\nClassification Report:")
print(classification_report(
    all_labels, all_preds,
    labels=list(range(len(test_dataset_aligned.classes))),
    target_names=test_dataset_aligned.classes,
    digits=4,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

#**Transfer Learning using pretrained weight**

In this step I am going to be using pretrained weights from the pytorch framework

In [ ]:
assert os.path.isdir(DATASET_128_ALIGNED_DIR), f"Dataset folder not found: {DATASET_128_ALIGNED_DIR}"
print("Device:", device)
print("Dataset:", DATASET_128_ALIGNED_DIR)

In [ ]:
IMG_SIZE = 128
BATCH_SIZE = 32

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.ToTensor(),
    normalize
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    normalize
])

train_dataset = datasets.ImageFolder(
    os.path.join(DATASET_128_ALIGNED_DIR, "train"),
    transform=train_transform
)

val_dataset = datasets.ImageFolder(
    os.path.join(DATASET_128_ALIGNED_DIR, "val"),
    transform=eval_transform
)

test_dataset = datasets.ImageFolder(
    os.path.join(DATASET_128_ALIGNED_DIR, "test"),
    transform=eval_transform
)

class_names = train_dataset.classes

assert train_dataset.class_to_idx == val_dataset.class_to_idx
assert train_dataset.class_to_idx == test_dataset.class_to_idx

loader_generator = torch.Generator().manual_seed(SEED)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=torch.cuda.is_available(), generator=loader_generator
)

val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available()
)

test_loader_aligned = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=torch.cuda.is_available()
)

print("Classes:", class_names)
print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))
print("Class mapping:", train_dataset.class_to_idx)

assert len(train_dataset) == 1766
assert len(val_dataset) == 367
assert len(test_dataset) == 367

In [ ]:
weights = models.ResNet18_Weights.DEFAULT

model_pretrained = models.resnet18(weights=weights)

# Replace the original 1,000-class layer with our five classes
model_pretrained.fc = nn.Linear(
    model_pretrained.fc.in_features,
    len(class_names)
)

model_pretrained = model_pretrained.to(device)

print("Pretrained ResNet18 loaded successfully.")
print("Output classes:", model_pretrained.fc.out_features)

In [ ]:
# Freeze all pretrained layers
for param in model_pretrained.parameters():
    param.requires_grad = False

# Train the new five-class output layer
for param in model_pretrained.fc.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    model_pretrained.fc.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

print(
    "Trainable parameters:",
    sum(p.numel() for p in model_pretrained.parameters()
        if p.requires_grad)
)

```markdown
### 4.9 Model Validation Loops & Best Save
```

In [ ]:
MAX_EPOCHS = 15
PATIENCE = 4

best_val_acc = -1.0
best_epoch = 0
ep_without_improve = 0
history = []

PRETRAINED_CHECKPOINT = os.path.join(PROJECT_DIR, "resnet18_128_pretrained_best.pth")
HISTORY_PATH = os.path.join(PROJECT_DIR, "resnet18_128_pretrained_history.csv")

start_time = time.time()

for epoch in range(MAX_EPOCHS):
    model_pretrained.train()
    model_pretrained.eval()
    model_pretrained.fc.train()

    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model_pretrained(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * images.size(0)
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    # Validation
    model_pretrained.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model_pretrained(images)
            loss = criterion(outputs, labels)

            val_loss_sum += loss.item() * images.size(0)
            val_correct += (outputs.argmax(dim=1) == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch [{epoch + 1:02d}/{MAX_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        ep_without_improve = 0
        torch.save(model_pretrained.state_dict(), PRETRAINED_CHECKPOINT)
    else:
        ep_without_improve += 1

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    if ep_without_improve >= PATIENCE:
        print("Early stopping.")
        break

print(f"\nBest validation accuracy: {best_val_acc:.2%}")
print("Best epoch:", best_epoch)
print(f"Training time: {(time.time() - start_time) / 60:.2f} minutes")
print("Checkpoint:", PRETRAINED_CHECKPOINT)
print("History:", HISTORY_PATH)

```markdown
### 4.10 Frozen Model Metrics Visualization
```

In [ ]:
model_pretrained.load_state_dict(
    torch.load(
        PRETRAINED_CHECKPOINT,
        map_location=device,
        weights_only=True
    )
)

model_pretrained.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader_aligned:
        images = images.to(device)

        outputs = model_pretrained(images)
        predictions = outputs.argmax(dim=1).cpu().numpy()

        all_preds.extend(predictions)
        all_labels.extend(labels.numpy())

test_accuracy = accuracy_score(all_labels, all_preds)

print(f"Pretrained ResNet18 Test Accuracy: {test_accuracy:.4%}")
print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_preds,
    labels=list(range(len(class_names))),
    target_names=class_names,
    digits=4,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

```markdown
## 5. Fine-Tuning Phase (Full Backpropagation)
### 5.1 Training Pre-checkpoint Load
```

In [ ]:
## Section 5: Fine-Tuning Phase (Full Backpropagation on 128x128)
### Subsection 5.1: Training Pre-checkpoint Load

##fine tuning transfer learning model

In [ ]:
PRETRAINED_CHECKPOINT = os.path.join(PROJECT_DIR, "resnet18_128_pretrained_best.pth")

model_finetuned = models.resnet18(weights=None)
model_finetuned.fc = nn.Linear(model_finetuned.fc.in_features, len(class_names))
model_finetuned.load_state_dict(torch.load(PRETRAINED_CHECKPOINT, map_location=device, weights_only=True))
model_finetuned = model_finetuned.to(device)

print("Loaded best frozen-backbone checkpoint.")
print("Device:", device)

### Subsection 5.2: Optimizer & Learning Rate Scheduling Configuration

In [ ]:
# Allow all model parameters to update
for param in model_finetuned.parameters():
    param.requires_grad = True

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.AdamW(
    [
        {
            "params": [
                p for name, p in model_finetuned.named_parameters()
                if not name.startswith("fc.")
            ],
            "lr": 1e-5
        },
        {
            "params": model_finetuned.fc.parameters(),
            "lr": 1e-4
        }
    ],
    weight_decay=1e-4
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="max",
    factor=0.5,
    patience=2
)

print("Backbone LR:", optimizer.param_groups[0]["lr"])
print("Classification head LR:", optimizer.param_groups[1]["lr"])
print(
    "Trainable parameters:",
    sum(p.numel() for p in model_finetuned.parameters()
        if p.requires_grad)
)

### Subsection 5.3: Active Fine-Tuning Execution

In [ ]:
MAX_EPOCHS = 15
PATIENCE = 4

best_val_acc = -1.0
best_epoch = 0
ep_without_improve = 0
finetune_history = []

FINETUNED_CHECKPOINT = os.path.join(PROJECT_DIR, "resnet18_128_finetuned_best.pth")
FINETUNED_HISTORY = os.path.join(PROJECT_DIR, "resnet18_128_finetuned_history.csv")

start_time = time.time()

for epoch in range(MAX_EPOCHS):
    model_finetuned.train()
    train_loss_sum = 0.0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        outputs = model_finetuned(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        train_loss_sum += loss.item() * images.size(0)
        train_correct += (outputs.argmax(dim=1) == labels).sum().item()
        train_total += labels.size(0)

    train_loss = train_loss_sum / train_total
    train_acc = train_correct / train_total

    # Validation
    model_finetuned.eval()
    val_loss_sum = 0.0
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)
            outputs = model_finetuned(images)
            loss = criterion(outputs, labels)

            val_loss_sum += loss.item() * images.size(0)
            val_correct += (outputs.argmax(dim=1) == labels).sum().item()
            val_total += labels.size(0)

    val_loss = val_loss_sum / val_total
    val_acc = val_correct / val_total

    scheduler.step(val_acc)
    current_lr_backbone = optimizer.param_groups[0]["lr"]
    current_lr_head = optimizer.param_groups[1]["lr"]

    finetune_history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "train_accuracy": train_acc,
        "val_loss": val_loss,
        "val_accuracy": val_acc,
        "backbone_lr": current_lr_backbone,
        "head_lr": current_lr_head
    })

    print(
        f"Epoch [{epoch + 1:02d}/{MAX_EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | "
        f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f} | "
        f"LR: {current_lr_backbone:.1e}/{current_lr_head:.1e}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        ep_without_improve = 0
        torch.save(model_finetuned.state_dict(), FINETUNED_CHECKPOINT)
    else:
        ep_without_improve += 1

    pd.DataFrame(finetune_history).to_csv(FINETUNED_HISTORY, index=False)

    if ep_without_improve >= PATIENCE:
        print("Early stopping.")
        break

print(f"\nBest validation accuracy: {best_val_acc:.2%}")
print("Best epoch:", best_epoch)
print(f"Training time: {(time.time() - start_time) / 60:.2f} minutes")
print("Checkpoint:", FINETUNED_CHECKPOINT)

### Subsection 5.4: Test Set Diagnostic Evaluation

In [ ]:
model_finetuned.load_state_dict(
    torch.load(
        FINETUNED_CHECKPOINT,
        map_location=device,
        weights_only=True
    )
)

model_finetuned.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for images, labels in test_loader_aligned:
        images = images.to(device)

        outputs = model_finetuned(images)
        predictions = outputs.argmax(dim=1).cpu().numpy()

        all_preds.extend(predictions)
        all_labels.extend(labels.numpy())

finetuned_accuracy = accuracy_score(all_labels, all_preds)

print(f"Fine-tuned Test Accuracy: {finetuned_accuracy:.4%}")

print("\nClassification Report:")
print(classification_report(
    all_labels,
    all_preds,
    labels=list(range(len(class_names))),
    target_names=class_names,
    digits=4,
    zero_division=0
))

print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_preds))

### Subsection 5.5: Consolidated Model Performance Review

In [ ]:
comparison = pd.DataFrame([
    {
        "Model": "ResNet18 from scratch",
        "Test Accuracy (%)": 95.10
    },
    {
        "Model": "Pretrained ResNet18 (frozen)",
        "Test Accuracy (%)": 95.37
    },
    {
        "Model": "Pretrained ResNet18 (fine-tuned)",
        "Test Accuracy (%)": round(finetuned_accuracy * 100, 2)
    }
])

display(comparison)

## Section 6: Visual Inspections & Location Metadata Verification
### Subsection 6.1: Individual Image Inference Verification

#Visual test of the best model

In [ ]:
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, 5)

checkpoint_path = os.path.join(PROJECT_DIR, "resnet18_128_finetuned_best.pth")
model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
model = model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

print("Model loaded successfully!")
print("Device:", device)

In [ ]:
test_dir = Path(os.path.join(DATASET_128_ALIGNED_DIR, "test"))

image_paths = [
    p for p in test_dir.rglob("*")
    if p.suffix.lower() in [".png", ".jpg", ".jpeg"]
]

image_path = random.choice(image_paths)
true_label = image_path.parent.name

image = Image.open(image_path).convert("RGB")
input_tensor = transform(image).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(input_tensor)
    probabilities = torch.softmax(logits, dim=1)[0]

predicted_index = probabilities.argmax().item()
predicted_label = class_names[predicted_index]
confidence = probabilities[predicted_index].item() * 100

plt.figure(figsize=(6, 6))
plt.imshow(image)
plt.axis("off")
plt.title(
    f"Predicted: {predicted_label}\n"
    f"Confidence: {confidence:.2f}%\n"
    f"Dataset label: {true_label}"
)
plt.show()

print("Prediction:", predicted_label)
print(f"Confidence: {confidence:.2f}%")
print("Dataset label:", true_label)
print("Correct prediction:", predicted_label == true_label)

### Subsection 6.2: Image Size & Nearest-Neighbor Resizing Evaluation

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

image = Image.open(image_path).convert("RGB")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

axes[0].imshow(image)
axes[0].set_title("Original: 128×128")
axes[0].axis("off")

axes[1].imshow(image, interpolation="nearest")
axes[1].set_title("Enlarged: nearest-neighbor")
axes[1].axis("off")

plt.tight_layout()
plt.show()

print("Image dimensions:", image.size)

### Subsection 6.3: Geographic Terrain Elevation Profile Verification

The sample is classified as Tree Cover, with 83.8% reference-label purity. Its location in the mountainous Tirah region is consistent with forested highland terrain. Elevation samples across the surrounding area vary by 357 m, indicating substantial local relief. However, the RGB image does not directly encode elevation, and this terrain information was not provided to the classifier

Google earth view for the tested image: https://earth.google.com/web/@33.763065,70.739110,2300a,3000d,35y,0h,0t,0r?utm_source=chatgpt.com

I have been able to assess that model is perfoming well.

In [ ]:
base = Path("/content/drive/MyDrive/landcover_project")

metadata_files = [
    base / "landcover_split_metadata.csv",
    base / "landcover_split_metadata_128.csv",
    base / "landcover_split_metadata_128_aligned.csv",
]

for path in metadata_files:
    if path.exists():
        df = pd.read_csv(path)
        print(f"\nFile: {path.name}")
        print("Columns:", df.columns.tolist())

        if "image_id" in df.columns:
            match = df[
                df["image_id"].astype(str).str.contains(
                    "01968", regex=False, na=False
                )
            ]

            if not match.empty:
                display(match)

In [ ]:
import requests
import numpy as np
import matplotlib.pyplot as plt

lat0 = 33.763065
lon0 = 70.739110

# Sample a 640 m × 640 m area around the image center.
# Approximately 0.00576° latitude and 0.0069° longitude
# span 640 m at this location.
latitudes = np.linspace(lat0 - 0.00288, lat0 + 0.00288, 5)
longitudes = np.linspace(lon0 - 0.00345, lon0 + 0.00345, 5)

grid_lat = np.repeat(latitudes, 5)
grid_lon = np.tile(longitudes, 5)

response = requests.get(
    "https://api.open-meteo.com/v1/elevation",
    params={
        "latitude": ",".join(map(str, grid_lat)),
        "longitude": ",".join(map(str, grid_lon))
    },
    timeout=30
)
response.raise_for_status()

elevations = np.array(response.json()["elevation"]).reshape(5, 5)

print(f"Minimum elevation: {elevations.min():.1f} m")
print(f"Maximum elevation: {elevations.max():.1f} m")
print(f"Elevation range:   {np.ptp(elevations):.1f} m")

plt.figure(figsize=(7, 5))
plt.imshow(
    elevations,
    origin="lower",
    extent=[-320, 320, -320, 320],
    aspect="equal"
)
plt.colorbar(label="Elevation (m)")
plt.xlabel("East–west distance from patch center (m)")
plt.ylabel("North–south distance from patch center (m)")
plt.title("Terrain elevation across the 640 m image patch")
plt.show()

In [ ]:
print("Elevation grid (metres):")
print(np.round(elevations, 0).astype(int))

plt.figure(figsize=(7, 5))
plt.plot(np.arange(5), elevations[2, :], marker="o", label="Middle row")
plt.plot(np.arange(5), elevations[:, 2], marker="s", label="Middle column")
plt.xlabel("Sample position across patch")
plt.ylabel("Elevation (m)")
plt.title("Elevation profiles through the image")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
plt.figure(figsize=(7, 5))

plt.imshow(
    elevations,
    cmap="terrain",
    origin="upper",
    interpolation="nearest"
)

plt.colorbar(label="Elevation (m)")
plt.title("Elevation pattern around img_01968")
plt.xlabel("West → East sample positions")
plt.ylabel("North → South sample positions")
plt.xticks(range(5))
plt.yticks(range(5))
plt.show()

### Subsection 6.4: Batch Random Inference & Earth Links Mapping

# Visual Test 2 with 15 images

In [ ]:
TEST_DIR = Path(os.path.join(DATASET_128_ALIGNED_DIR, "test"))
METADATA_PATH = os.path.join(PROJECT_DIR, "landcover_split_metadata.csv")
CHECKPOINT_PATH = os.path.join(PROJECT_DIR, "resnet18_128_finetuned_best.pth")

# ==========================================
# 1. LOAD MODEL
# ==========================================
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(class_names))

model.load_state_dict(
    torch.load(
        CHECKPOINT_PATH,
        map_location=device,
        weights_only=True
    )
)

model.to(device)
model.eval()

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

# ==========================================
# 2. LOAD COORDINATE METADATA
# ==========================================
metadata = pd.read_csv(METADATA_PATH)
metadata["image_id"] = metadata["image_id"].astype(str)

metadata_lookup = metadata.set_index("image_id").to_dict("index")

# ==========================================
# 3. SELECT 3 RANDOM TEST IMAGES PER CLASS
# ==========================================
random.seed(SEED)
selected_images = []

for class_name in class_names:
    class_dir = TEST_DIR / class_name

    image_files = [
        p for p in class_dir.iterdir()
        if p.suffix.lower() in [".png", ".jpg", ".jpeg"]
    ]

    # Only use images with matching coordinate metadata
    image_files = [
        p for p in image_files
        if p.stem in metadata_lookup
    ]

    if len(image_files) < 3:
        raise ValueError(
            f"Found only {len(image_files)} usable images for {class_name}"
        )

    selected_images.extend(
        random.sample(image_files, 3)
    )

# ==========================================
# 4. RUN INFERENCE AND DISPLAY GALLERY
# ==========================================
results = []
fig, axes = plt.subplots(3, 5, figsize=(20, 13))
axes = axes.flatten()

for i, image_path in enumerate(selected_images):
    image = Image.open(image_path).convert("RGB")
    input_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        logits = model(input_tensor)
        probs = torch.softmax(logits, dim=1)[0]

    pred_idx = probs.argmax().item()
    predicted_class = class_names[pred_idx]
    confidence = probs[pred_idx].item() * 100

    true_class = image_path.parent.name
    correct = predicted_class == true_class

    row = metadata_lookup[image_path.stem]
    latitude = float(row["Latitude"])
    longitude = float(row["Longitude"])

    earth_url = (
        "https://earth.google.com/web/"
        f"@{latitude},{longitude},2300a,2500d,35y,0h,0t,0r"
    )

    axes[i].imshow(image)
    axes[i].axis("off")
    axes[i].set_title(
        f"{image_path.stem}\n"
        f"Pred: {predicted_class.replace('_', ' ')}\n"
        f"Confidence: {confidence:.1f}%\n"
        f"True: {true_class.replace('_', ' ')}\n"
        f"{'CORRECT' if correct else 'MISMATCH'}",
        fontsize=9,
        color="green" if correct else "red"
    )

    results.append({
        "Image": image_path.name,
        "Reference class": true_class,
        "Predicted class": predicted_class,
        "Confidence (%)": round(confidence, 2),
        "Correct": correct,
        "Latitude": latitude,
        "Longitude": longitude,
        "Google Earth URL": earth_url
    })

plt.suptitle(
    "Land-Cover Model: 15 Random Test Images",
    fontsize=17
)
plt.tight_layout()
plt.show()

# ==========================================
# 5. PRINT COORDINATES AND GOOGLE EARTH LINKS
# ==========================================
results_df = pd.DataFrame(results)

print("Visual inspection results:")
display(results_df.drop(columns=["Google Earth URL"]))

print("\nGoogle Earth links — click to inspect each location:\n")

for _, row in results_df.iterrows():
    print(
        f"{row['Image']} | {row['Reference class']} | "
        f"Predicted: {row['Predicted class']} | "
        f"Confidence: {row['Confidence (%)']}%"
    )
    print(f"Coordinates: {row['Latitude']}, {row['Longitude']}")
    print(row["Google Earth URL"])
    print("-" * 90)

output_csv = os.path.join(PROJECT_DIR, "visual_inspection_15_images.csv")
results_df.to_csv(output_csv, index=False)

print(f"\nResults saved to: {output_csv}")
print(f"Images inspected: {len(results_df)}")
print(f"Correct predictions: {results_df['Correct'].sum()}/15")

## Section 7: Model Archival & Client Deployment
### Subsection 7.1: Model Weights & Config Serialization

In [ ]:
CHECKPOINT_PATH = os.path.join(PROJECT_DIR, "resnet18_128_finetuned_best.pth")

EXPORT_DIR = Path("/content/landcover_model_export")
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

assert os.path.exists(CHECKPOINT_PATH), f"Checkpoint not found: {CHECKPOINT_PATH}"

# ==========================================
# 1. Load the trained checkpoint
# ==========================================
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

if isinstance(checkpoint, dict):
    if "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    elif "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    else:
        state_dict = checkpoint
else:
    raise TypeError("Unexpected checkpoint format.")

state_dict = {
    key.removeprefix("module."): value
    for key, value in state_dict.items()
}

# ==========================================
# 2. Reconstruct the model and verify weights
# ==========================================
model = models.resnet18(weights=None)
model.fc = torch.nn.Linear(model.fc.in_features, 5)
model.load_state_dict(state_dict, strict=True)
model.eval()

weights_path = EXPORT_DIR / "model_weights.pth"
torch.save(model.state_dict(), weights_path)

# ==========================================
# 3. Save class names and preprocessing config
# ==========================================
config = {
    "model_architecture": "ResNet18",
    "num_classes": 5,
    "class_names": classes,
    "input_size": [128, 128],
    "input_channels": 3,
    "input_color_order": "RGB",
    "input_value_range_before_normalization": [0, 1],
    "normalization": {
        "mean": [0.485, 0.456, 0.406],
        "std": [0.229, 0.224, 0.225]
    },
    "checkpoint_source": "resnet18_128_finetuned_best.pth",


In [ ]:

from google.colab import files

files.download("/content/landcover_model_export.zip")
